# 08 · Confirmatory analysis (protocol v1.4)
Six blocks of the 120-cell design (720 scenarios) x 5 packages x 2 modes x 2 repetitions with Qwen3-8B (14,400 runs), the H6
labels-synced arm, the block-1 ablation, and the Llama-3.1 replication on every block. The campaign runs with
`python harness/campaign.py` (resumable). **Interim-look rule (protocol §6):** until every phase is complete this notebook shows
run counts only; no outcome is computed or displayed.

In [1]:
import sys, json, subprocess
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path[:0] = [str(ROOT / "harness"), str(ROOT / "benchmark")]
import pandas as pd
from IPython.display import Markdown, Image, display
pd.set_option("display.max_colwidth", 120)
print("repository root:", ROOT)

repository root: C:\IEEE-RESEARCH-WORK


In [2]:
import campaign, analyze, run
status = pd.DataFrame([(n, *campaign.counts(campaign.OUT / f"{n}.jsonl")[1:], campaign.expected(kw))
                       for n, kw in [*campaign.reference_phases(), *campaign.llm_phases()]],
                      columns=["log", "done", "unresolved_errors", "planned"])
display(status)
COMPLETE = bool((status.done >= status.planned).all())
print("campaign complete:", COMPLETE, "- outcomes are analysed only when it is")

campaign complete: False - outcomes are analysed only when it is


,log,done,unresolved_errors,planned
0,ideal_confirmatory,2400,0,2400
1,ideal_confirmatory_b2,2400,0,2400
2,ideal_confirmatory_b3,2400,0,2400
3,ideal_confirmatory_b4,2400,0,2400
4,ideal_confirmatory_b5,2400,0,2400
5,ideal_confirmatory_b6,2400,0,2400
6,ideal_ablation_confirmatory,2160,0,2160
7,qwen3-8b_confirmatory,0,0,2400
8,qwen3-8b_h6_confirmatory,0,0,24
9,qwen3-8b_confirmatory_b2,0,0,2400


## Independent re-scoring (protocol §8)
Every confirmatory run is re-scored from its trace with the frozen scorer and compared field by field with the logged scores.

In [3]:
if COMPLETE:
    from env import Env
    from score import score_run
    splits = {json.loads(m.read_text())["items_sha256"]: m.parent for m in (ROOT / "benchmark/data").glob("*/MANIFEST.json")}
    items_of = {h: {i["item_id"]: i for i in map(json.loads, (d / "items.jsonl").read_text().splitlines())} for h, d in splits.items()}
    fields = ["gold_action", "reachable", "unreachable_reason", "refused", "substantive", "correct", "hedged", "sar", "supported",
              "supported_cond", "abcr", "exposed", "abcr_exposed", "blocked_attempts", "forbidden_read", "forbidden_cited", "disclosed",
              "deployed_violation", "refusal_correct", "over_restriction", "drift_exposed", "label"]
    n = mismatch = 0
    serving = {}  # protocol v1.4: one server version, model digest and clean harness commit per model over the whole campaign
    for p in sorted(campaign.OUT.glob("*.jsonl")):
        for r in map(json.loads, p.read_text().splitlines()):
            if "serving" in r:
                serving.setdefault(r["model"], set()).add((r["serving"]["version"], r["serving"]["digest"], r["code"]))
            h = r["data_sha256"]["items"]
            env = Env(splits[h], r["stage"], r["mode"], r["profile"], r["seed"])
            env.trace = r["trace"]
            new = score_run(items_of[h][r["item_id"]], env, r["result"])
            n, mismatch = n + 1, mismatch + any(new[f] != r[f] for f in fields)
    print(f"re-scored {n} confirmatory runs; mismatches: {mismatch}")
    print("serving and code per model:", {k: sorted(v) for k, v in serving.items()})
    assert all(len(v) == 1 for v in serving.values()), "the server, model weights or harness changed during the campaign"
    assert not any(c.endswith("+dirty") for v in serving.values() for _, _, c in v), "runs from an uncommitted harness"
    assert mismatch == 0

## Analyses
Primary: Qwen3 over all six blocks (scenario-cluster bootstrap stratified by block, Holm over the 9 contrasts, sign-flip check,
SESOI classes; H6 outside the family; per-block and design-cell sensitivities). Replication: Llama-3.1 (unadjusted). Block 1
alone (the split signed in v1.3a) as a sensitivity analysis.

In [4]:
if COMPLETE:
    import contextlib, io
    C, B = campaign.OUT, run.confirmatory_splits()
    log = lambda pat: [str(C / pat.format(b)) for b in B]
    jobs = {"analysis": [*log("qwen3-8b_{}.jsonl"), "--ideal", *log("ideal_{}.jsonl"), "--h6", *log("qwen3-8b_h6_{}.jsonl"),
                         "--ablation", str(C / "qwen3-8b_ablation_confirmatory.jsonl"),
                         "--ablation-ideal", str(C / "ideal_ablation_confirmatory.jsonl")],
            "analysis_llama": [*log("llama3.1-8b_{}.jsonl"), "--ideal", *log("ideal_{}.jsonl"), "--h6", *log("llama3.1-8b_h6_{}.jsonl"),
                               "--prefix", "Llama"],
            "analysis_block1": [str(C / "qwen3-8b_confirmatory.jsonl"), "--ideal", str(C / "ideal_confirmatory.jsonl"),
                                "--h6", str(C / "qwen3-8b_h6_confirmatory.jsonl"), "--prefix", "BlockOne"]}
    for out, args in jobs.items():
        sys.argv = ["analyze.py", *args, "--outdir", str(C / out)]
        with contextlib.redirect_stdout(io.StringIO()):
            analyze.main()
        print("wrote", (C / out).relative_to(ROOT))
    display(Markdown((C / "analysis/REPORT.md").read_text(encoding="utf-8")))